# Preprocess half hourly flux data
* data loading
* data cleaning

In [1]:
import pandas as pd
import plotnine as p9
import numpy as np

In [2]:
harv = pd.read_csv('../data/full/AMF_US-xHA_FLUXNET_FULLSET_HH_2019-2024_4-7.csv')
stei = pd.read_csv('../data/full/AMF_US-xST_FLUXNET_FULLSET_HH_2019-2024_4-7.csv')

In [3]:
def data_cleaning(df):
    cols = ['TIMESTAMP_START','TIMESTAMP_END','TA_F','TA_F_QC','SW_IN_F','SW_IN_F_QC','VPD_F','VPD_F_QC', 'P_F','P_F_QC','PPFD_IN','PPFD_OUT','CO2_F_MDS','CO2_F_MDS_QC','TS_F_MDS_1','TS_F_MDS_2','TS_F_MDS_3','TS_F_MDS_4','TS_F_MDS_6','TS_F_MDS_1_QC','TS_F_MDS_2_QC','TS_F_MDS_3_QC','TS_F_MDS_4_QC','TS_F_MDS_6_QC','SWC_F_MDS_1','SWC_F_MDS_2','SWC_F_MDS_3',
            'SWC_F_MDS_5','SWC_F_MDS_6','SWC_F_MDS_1_QC','SWC_F_MDS_2_QC',
            'SWC_F_MDS_3_QC','SWC_F_MDS_5_QC','SWC_F_MDS_6_QC','NEE_VUT_REF','NEE_VUT_REF_QC','RECO_NT_VUT_REF','GPP_NT_VUT_REF','RECO_DT_VUT_REF','GPP_DT_VUT_REF']
    df_sub = df[cols]

    # data cleaning
    print("Initial data counts: {} records".format(len(df_sub)))

    df_clean = df_sub[df_sub['NEE_VUT_REF']!=-9999]
    print("Removed gaps in NEE: {} records left".format(len(df_clean)))

    df_clean = df_sub[df_sub['NEE_VUT_REF_QC']<=1]
    print("Removed low-quality NEE data: {} records left".format(len(df_clean)))

    df_clean = df_clean[df_clean['TA_F_QC']<=1]
    print("Removed low quality TA data: {} records left".format(len(df_clean)))

    df_clean = df_clean[df_clean['SW_IN_F_QC']<=1]
    print("Removed low quality SW_IN data: {} records left".format(len(df_clean)))

    df_clean = df_clean[df_clean['VPD_F_QC']<=1]
    print("Removed low quality VPD data: {} records left".format(len(df_clean)))

    df_clean = df_clean[df_clean['P_F_QC']<=1]
    print("Removed low quality P data: {} records left".format(len(df_clean)))

    df_clean = df_clean[df_clean['GPP_NT_VUT_REF']>=-5]
    print("Removed large negative GPP data: {} records left".format(len(df_clean)))

    df_clean = df_clean.replace({-9999:np.nan})

    # parse time
    df_clean['datetime'] = pd.to_datetime(df_clean['TIMESTAMP_START'], format="%Y%m%d%H%M")
    df_clean["year"] = df_clean['datetime'].dt.year
    df_clean["month"] = df_clean['datetime'].dt.month
    df_clean["day"] = df_clean['datetime'].dt.day
    df_clean["time"] = df_clean['datetime'].dt.strftime("%H:%M")
    df_clean.head()

    return df_clean


In [4]:
harv_clean = data_cleaning(harv)

Initial data counts: 105216 records
Removed gaps in NEE: 105216 records left
Removed low-quality NEE data: 85888 records left
Removed low quality TA data: 85887 records left
Removed low quality SW_IN data: 85884 records left
Removed low quality VPD data: 83393 records left
Removed low quality P data: 81987 records left
Removed large negative GPP data: 81705 records left


In [5]:
stei_clean = data_cleaning(stei)

Initial data counts: 105216 records
Removed gaps in NEE: 105216 records left
Removed low-quality NEE data: 89380 records left
Removed low quality TA data: 89377 records left
Removed low quality SW_IN data: 89314 records left
Removed low quality VPD data: 89304 records left
Removed low quality P data: 79670 records left
Removed large negative GPP data: 79631 records left


In [6]:
harv_clean.to_csv('../data/harv_clean.csv', index=False)
stei_clean.to_csv('../data/stei_clean.csv', index=False)